# RK1828 SenseVoice-Small Multilingual ASR Demo


Alibaba [SenseVoice-Small](https://github.com/FunAudioLLM/SenseVoice)
multilingual ASR. Besides the transcript, the output **inlines language,
emotion and sound-event tags** (e.g. `<|zh|><|HAPPY|><|Speech|>`). Measured
~292 ms per clip.

| Step | Content |
|---|---|
| 1 | Download models and sample audio (from huggingface.co) |
| 2 | Extract the prebuilt demo |
| 3 | Pick an audio clip |
| 4 | Transcribe (see the rich tags) |

## 0. Environment check

Make sure the RKNN3 runtime (`librknn3_api.so`) and the RK1828 device are ready.
If this fails, install the driver/runtime first (the `rknn3-rk182x` DEB package)
and check with `sudo rknn-smi info` that the RK1828 shows up.

In [ ]:
from rknn3 import get_lib, find_devices

_ = get_lib()  # loads /usr/lib/librknn3_api.so; failure means the runtime is not installed

devices = find_devices()
if not devices:
    raise RuntimeError("No RK1828 device found! Check: sudo rknn-smi info")
for dev_id, dev_type in devices:
    print(f"Found device: id={dev_id}  interface={dev_type}")

## 1. Download models

The models for this demo are hosted on Hugging Face (account `qhddz`) and are
downloaded directly from **huggingface.co**. Run the next cell to start.

> Files that already exist (and are non-empty) are **skipped automatically** —
> re-running never downloads them twice.
> If your network cannot reach huggingface.co, set up a proxy first, e.g.
> `export https_proxy=http://<your-proxy>:<port>` before starting Jupyter.

In [ ]:
# (HF repo, path in repo, local path)
FILES = [
    ("sensevoice-small-rknn3-rk1828", "demo_x86_64.tar.gz", "demos/sensevoice/demo_x86_64.tar.gz"),
    ("sensevoice-small-rknn3-rk1828", "model/embedding_matrix.txt", "demos/sensevoice/model/embedding_matrix.txt"),
    ("sensevoice-small-rknn3-rk1828", "model/sensevoice_fix_10s.rknn", "demos/sensevoice/model/sensevoice_fix_10s.rknn"),
    ("sensevoice-small-rknn3-rk1828", "model/sensevoice_fix_10s.weight", "demos/sensevoice/model/sensevoice_fix_10s.weight"),
    ("sensevoice-small-rknn3-rk1828", "model/tokens.txt", "demos/sensevoice/model/tokens.txt"),
    ("sensevoice-small-rknn3-rk1828", "model/zh.wav", "demos/sensevoice/model/zh.wav"),
    ("whisper-base-rknn3-rk1828", "model/test_zh.wav", "datasets/asr/test_zh.wav"),
    ("whisper-base-rknn3-rk1828", "model/test_en.wav", "datasets/asr/test_en.wav"),
    ("whisper-base-rknn3-rk1828", "model/long_en.wav", "datasets/asr/long_en.wav"),
    ("sensevoice-small-rknn3-rk1828", "model/zh.wav", "datasets/asr/sensevoice_zh.wav"),
    ("paraformer-rknn3-rk1828", "model/asr_example.wav", "datasets/asr/asr_example.wav"),
    ("paraformer-rknn3-rk1828", "model/asr_example_hotword.wav", "datasets/asr/asr_example_hotword.wav"),
    ("qwen3-forced-aligner-rknn3-rk1828", "audio.wav", "datasets/asr/aligner_demo.wav"),
]

In [ ]:
from nb_download import ensure_files

ensure_files(FILES, "hf")

## 2. Extract the prebuilt demo

`demo_x86_64.tar.gz` contains the official x86_64 C++ demo and its runtime
libraries (`lib/`). Hugging Face does not host raw executables, so they are
shipped as a tarball; extract once (re-running skips automatically).

> **Why a C++ demo instead of pure Python?**
> Notebooks 01-05 (classification / detection / FastSAM / pose / segmentation)
> are a single model with one forward pass and simple post-processing, so they
> walk through everything in Python (`rknn3.py`). This demo's pipeline is far
> more complex — chained models, tokenizer, KV cache, sliding-window /
> streaming state, hundreds of lines of orchestration — reimplementing it in
> Python would be costly and risk diverging from the official behavior.
> So we run the official `rknn3-model-zoo` C++ demo directly; source code:
> [github.com/airockchip/rknn3-model-zoo](https://github.com/airockchip/rknn3-model-zoo)
> under `examples/`.

In [ ]:
from nb_download import extract_tarball

extract_tarball("demos/sensevoice/demo_x86_64.tar.gz", "demos/sensevoice")

In [ ]:
from nb_demo import run_demo, snapshot, new_files, show_images, show_new_images, show_audio

## 3. Pick an audio clip

In [ ]:
import glob
import ipywidgets as widgets
from IPython.display import display

w_wav = widgets.Dropdown(options=sorted(glob.glob("datasets/asr/*.wav")),
                         description="Audio:", layout=widgets.Layout(width="60%"))
display(w_wav)
if any(o.endswith('/sensevoice_zh.wav') for o in w_wav.options):
    w_wav.value = [o for o in w_wav.options if o.endswith('/sensevoice_zh.wav')][0]
print("Then keep running downwards ▼")

Listen to the audio first:

In [ ]:
from nb_demo import show_audio

show_audio(w_wav.value)

## 4. Transcribe

In [ ]:
import os

DEMO = "demos/sensevoice"
wav = os.path.abspath(w_wav.value)
run_demo(["./rknn_sensevoice_demo",
          "model/sensevoice_fix_10s.rknn", "model/sensevoice_fix_10s.weight",
          wav, "0x1"], DEMO)

## Summary

- SenseVoice generates "language/emotion/event" as special tokens, so one
  inference gives you the transcript plus paralinguistic info.
- The model input is fixed at 10 s; shorter audio is padded automatically.